# FiftyOne Embeddings Panel 可视化

本 notebook 演示如何使用 FiftyOne 的 **Embeddings Panel** 功能来可视化自定义的 embeddings 数据。

## 功能说明

- 使用自定义的 embeddings（无需通过模型计算）
- 在 FiftyOne App 中使用 Embeddings Panel 进行交互式可视化
- 支持按标签着色、选择样本、过滤等操作
- 可以识别异常样本和聚类结构

## 1. 导入必要的库

In [1]:
import os
import os.path as osp
import numpy as np
import cv2
from tqdm import tqdm

import fiftyone as fo
import fiftyone.brain as fob

## 2. 准备数据

**数据格式说明：**
- `image_paths`: 图像文件路径列表（字符串或 numpy 数组）
- `mask_paths`: 掩码文件路径列表（字符串或 numpy 数组）
- `label_names`: 标签名称列表（字符串或 numpy 数组）
- `embeddings`: embeddings 数组，形状为 (N, D)，其中 N 是样本数，D 是 embedding 维度

**注意：** 请根据你的实际数据来源修改下面的代码。

In [2]:
# ============================================
# 方式 1: 从 NPZ 文件加载数据
# ============================================
npz_path = "../checkpoints/supcon_models/1223-2/extract_embeddings/4.x_val.npz"
data = np.load(npz_path)
embeddings = data['embeddings']
label_names = data['label_names']
image_paths = data['image_paths']
mask_paths = data['mask_paths']

# ============================================
# 方式 2: 直接使用已有的变量
# ============================================
# 如果你已经在 notebook 中有了这些变量，可以直接使用
# 确保变量名如下：
# - image_paths
# - mask_paths  
# - label_names
# - embeddings

# ============================================
# 方式 3: 从其他格式加载
# ============================================
# 根据你的实际情况修改下面的代码
# image_paths = [...]  # 你的图像路径列表
# mask_paths = [...]   # 你的掩码路径列表
# label_names = [...]  # 你的标签列表
# embeddings = np.array([...])  # 你的 embeddings 数组

# 示例：打印数据形状
print(f"Embeddings 形状: {embeddings.shape}")
print(f"样本数量: {len(image_paths)}")
print(f"标签数量: {len(set(label_names))}")
print(f"唯一标签: {list(set(label_names))[:10]}...")  # 显示前10个标签

Embeddings 形状: (207, 128)
样本数量: 207
标签数量: 20
唯一标签: [np.str_('笔迹'), np.str_('装配错位-矩形极柱(全局)'), np.str_('压伤'), np.str_('异物-杂物、灰尘、黑线'), np.str_('异物-人造'), np.str_('划伤-正常'), np.str_('碰伤'), np.str_('破损'), np.str_('异物遮蔽'), np.str_('划伤-重度')]...


## 筛选特定标签的样本

In [3]:
# selected_label = "cable"  # 替换为你想筛选的标签
# # 筛选特定标签的样本
# filtered_image_paths = []
# filtered_mask_paths = []
# filtered_embeddings = []
# filtered_label_names = []
# for img_path, mask_path, emb, label in zip(image_paths, mask_paths, embeddings, label_names):
#     if selected_label in label:
#         filtered_image_paths.append(img_path)
#         filtered_mask_paths.append(mask_path)
#         filtered_embeddings.append(emb)
#         filtered_label_names.append(label)

# # 转换为 numpy 数组
# embeddings = np.array(filtered_embeddings)
# image_paths = filtered_image_paths
# mask_paths = filtered_mask_paths
# label_names = filtered_label_names

# print(f"Embeddings 形状: {embeddings.shape}")
# print(f"样本数量: {len(image_paths)}")
# print(f"标签数量: {len(set(label_names))}")
# print(f"唯一标签: {list(set(label_names))[:10]}...")  # 显示前10个标签

## 3. 数据预处理

将数据转换为 FiftyOne 可以使用的格式。

In [4]:
# 确保 embeddings 是 numpy 数组
if not isinstance(embeddings, np.ndarray):
    embeddings = np.array(embeddings)

# 确保所有路径都是字符串格式
def decode_if_bytes(x):
    """如果是 bytes 类型，解码为字符串"""
    if isinstance(x, bytes):
        return x.decode('utf-8')
    return str(x)

# 转换路径格式
image_paths = [decode_if_bytes(p) for p in image_paths]
mask_paths = [decode_if_bytes(p) for p in mask_paths]
label_names = [decode_if_bytes(l) for l in label_names]

# 验证数据一致性
assert len(image_paths) == len(mask_paths) == len(label_names) == len(embeddings), \
    f"数据长度不一致: images={len(image_paths)}, masks={len(mask_paths)}, labels={len(label_names)}, embeddings={len(embeddings)}"

print(f"✓ 数据预处理完成")
print(f"  - 样本数: {len(image_paths)}")
print(f"  - Embedding 维度: {embeddings.shape[1]}")
print(f"  - 唯一标签数: {len(set(label_names))}")

✓ 数据预处理完成
  - 样本数: 207
  - Embedding 维度: 128
  - 唯一标签数: 20


## 4. 创建 FiftyOne 样本

为每个图像创建 `fo.Sample` 对象，添加图像、掩码和标签信息。

In [5]:
# 设置图像路径的基础目录（如果需要）
# 如果你的 image_paths 和 mask_paths 是相对路径，需要设置基础目录
# BASE_DIR = os.getcwd()  # 或者设置为你的数据根目录
BASE_DIR = "/home/unitx/workspace_custom/EmbeddingModel"

samples = []
samples_embeddings = []

print("正在创建 FiftyOne 样本...")
for i, (img_path, mask_path, label, emb) in enumerate(tqdm(
    zip(image_paths, mask_paths, label_names, embeddings), 
    total=len(image_paths),
    desc="创建样本"
)):
    # 处理图像路径
    if not osp.isabs(img_path):
        img_path = osp.join(BASE_DIR, img_path)
    
    # 处理掩码路径
    if not osp.isabs(mask_path):
        mask_path = osp.join(BASE_DIR, mask_path)
    
    # 检查文件是否存在
    if not osp.exists(img_path):
        print(f"警告: 图像文件不存在，跳过: {img_path}")
        continue
    
    if not osp.exists(mask_path):
        print(f"警告: 掩码文件不存在，跳过: {mask_path}")
        continue
    
    # 创建样本
    sample = fo.Sample(filepath=img_path)
    
    # 添加分类标签
    sample["ground_truth"] = fo.Classification(label=label)
    
    # 添加分割掩码（如果存在）
    try:
        mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask_image is not None:
            sample["segmentation"] = fo.Segmentation(mask=mask_image, tags=[label])
    except Exception as e:
        print(f"警告: 无法加载掩码 {mask_path}: {e}")
    
    samples.append(sample)
    samples_embeddings.append(emb)

# 将 embeddings 转换为 numpy 数组
samples_embeddings = np.stack(samples_embeddings)

print(f"\n✓ 成功创建 {len(samples)} 个样本")
print(f"  - Embeddings 形状: {samples_embeddings.shape}")

正在创建 FiftyOne 样本...


创建样本: 100%|██████████| 207/207 [00:00<00:00, 1490.69it/s]


✓ 成功创建 207 个样本
  - Embeddings 形状: (207, 128)


## 5. 创建 FiftyOne 数据集

In [6]:
# 设置数据集名称
dataset_name = "embeddings-dataset"

# 删除已存在的同名数据集（可选）
if fo.dataset_exists(dataset_name):
    fo.delete_dataset(dataset_name)
    print(f"删除已存在的数据集: {dataset_name}")

# 创建新数据集
dataset = fo.Dataset(dataset_name)
dataset.add_samples(samples)

print(f"✓ 成功创建数据集: {dataset_name}")
print(f"  - 样本数: {len(dataset)}")
print(f"  - 数据集大小: {dataset.stats()}")

You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
删除已存在的数据集: embeddings-dataset
 100% |█████████████████| 207/207 [408.5ms elapsed, 0s remaining, 507.7 samples/s]      
✓ 成功创建数据集: embeddings-dataset
  - 样本数: 207
  - 数据集大小: {'samples_count': 207, 'samples_bytes': 289994, 'samples_size': '283.2KB', 'total_bytes': 289994, 'total_size': '283.2KB'}


## 6. 计算可视化（使用自定义 Embeddings）

使用 `fob.compute_visualization()` 计算 2D 可视化。这个方法会：
1. 使用 UMAP 将高维 embeddings 降维到 2D
2. 将结果保存到数据集的 brain key 中
3. 可以在 Embeddings Panel 中使用这个 brain key

In [7]:
# 设置 brain key（在 Embeddings Panel 中会用到这个 key）
print("正在计算可视化...")
print("这可能需要几分钟时间，取决于样本数量...")

# 计算可视化
# 使用自定义的 embeddings，而不是通过模型计算
mothed = "umap"   # umap / tsne

if mothed == "umap":
    brain_key = "custom_embeddings_umap"
elif mothed == "tsne":
    brain_key = "custom_embeddings_tsne"
else:
    raise ValueError(f"未知的降维方法: {mothed}")

# 清理已有的同名 brain results
if brain_key in dataset.list_brain_runs():
    print(f"发现已存在的 brain key: {brain_key}，正在删除...")
    dataset.delete_brain_run(brain_key)
    print(f"✓ 已删除旧的 brain results: {brain_key}")

# 计算可视化
if mothed == "umap":
    results = fob.compute_visualization(
        dataset,
        embeddings=samples_embeddings,  # 使用自定义 embeddings
        num_dims=2,                      # 降维到 2D
        method="umap",                   # 使用 UMAP 方法
        seed=51,                         # 随机种子，保证结果可复现
        brain_key=brain_key,             # 保存结果的 key
        verbose=True,                    # 显示进度信息
    )
elif mothed == "tsne":
    results = fob.compute_visualization(
        dataset,
        embeddings=samples_embeddings,  # 使用自定义 embeddings
        num_dims=2,                      # 降维到 2D
        method="tsne",                   # 使用 TSNE 方法
        seed=51,                         # 随机种子，保证结果可复现
        brain_key=brain_key,             # 保存结果的 key
        verbose=True,                    # 显示进度信息
    )

print(f"\n✓ 可视化计算完成")
print(f"  - Brain key: {brain_key}")
print(f"  - 可以在 Embeddings Panel 中使用这个 key 来查看可视化结果")

正在计算可视化...
这可能需要几分钟时间，取决于样本数量...
Generating visualization...


/home/unitx/miniconda3/envs/hjh/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP(n_jobs=1, random_state=51, verbose=True)
Wed Dec 24 09:34:08 2025 Construct fuzzy simplicial set
Wed Dec 24 09:34:08 2025 Finding Nearest Neighbors
Wed Dec 24 09:34:11 2025 Finished Nearest Neighbor Search
Wed Dec 24 09:34:13 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Wed Dec 24 09:34:13 2025 Finished embedding

✓ 可视化计算完成
  - Brain key: custom_embeddings_umap
  - 可以在 Embeddings Panel 中使用这个 key 来查看可视化结果


## 7. 启动 FiftyOne App 并使用 Embeddings Panel

启动 FiftyOne 应用程序，然后在界面中使用 Embeddings Panel 进行可视化。

In [8]:
# 启动 FiftyOne 应用
print("正在启动 FiftyOne 应用...")
session = fo.launch_app(dataset, port=5151)
print(f"✓ FiftyOne 应用已启动")
print(f"  - 访问地址: http://localhost:5151")
print(f"\n📌 使用 Embeddings Panel 的步骤：")
print(f"  1. 在 FiftyOne App 界面中，点击左上角的 '+' 按钮")
print(f"  2. 选择 'New panel > Embeddings'")
print(f"  3. 在 'Select brain key' 下拉菜单中选择: {brain_key}")
print(f"  4. 现在你应该能看到 2D 的 embeddings 可视化")
print(f"\n💡 Embeddings Panel 功能：")
print(f"  - 使用鼠标拖拽选择区域（lasso tool）来选择样本")
print(f"  - 使用 'Color by' 选项按标签或其他字段着色")
print(f"  - 选中的样本会在主界面中高亮显示")
print(f"  - 可以识别异常样本和聚类结构")

正在启动 FiftyOne 应用...


✓ FiftyOne 应用已启动
  - 访问地址: http://localhost:5151

📌 使用 Embeddings Panel 的步骤：
  1. 在 FiftyOne App 界面中，点击左上角的 '+' 按钮
  2. 选择 'New panel > Embeddings'
  3. 在 'Select brain key' 下拉菜单中选择: custom_embeddings_umap
  4. 现在你应该能看到 2D 的 embeddings 可视化

💡 Embeddings Panel 功能：
  - 使用鼠标拖拽选择区域（lasso tool）来选择样本
  - 使用 'Color by' 选项按标签或其他字段着色
  - 选中的样本会在主界面中高亮显示
  - 可以识别异常样本和聚类结构


## 8. （可选）在 Notebook 中直接可视化

如果你想在 notebook 中直接查看可视化结果，可以使用下面的代码：

In [9]:
# 获取标签用于着色
labels = [s["ground_truth"].label for s in samples]

# 创建可视化图表
try:
    plot = results.visualize(labels=labels)
    
    # 方法 1: 直接显示图表（如果出现 widget 错误，可以跳过）
    try:
        plot.show(height=720, width=1280)
    except Exception as e:
        print(f"警告: 无法显示交互式图表: {e}")
        print("提示: 你仍然可以在 FiftyOne App 的 Embeddings Panel 中查看可视化结果")
    
    # 方法 2: 将图表附加到 session（在 notebook 中可以使用 lasso 工具选择样本）
    try:
        if 'session' in globals() and session is not None:
            session.plots.attach(plot)
            print("✓ 图表已附加到 session")
    except Exception as e:
        print(f"警告: 无法附加图表到 session: {e}")
        print("提示: 你仍然可以在 FiftyOne App 的 Embeddings Panel 中查看可视化结果")
        
except Exception as e:
    print(f"错误: 无法创建可视化图表: {e}")
    print("提示: 请确保已经运行了步骤 6（计算可视化）和步骤 7（启动 App）")

FigureWidget({
    'data': [{'customdata': array(['694b430ed025705979bd9b87', '694b430ed025705979bd9b88'], dtype=object),
              'hovertemplate': ('<b>label: %{text}</b><br>x, y ' ... ': %{customdata}<extra></extra>'),
              'line': {'color': '#AA0DFE'},
              'mode': 'markers',
              'name': np.str_('R角凸起'),
              'showlegend': True,
              'text': array(['R角凸起', 'R角凸起'], dtype=object),
              'type': 'scattergl',
              'uid': 'b8eac26b-9e7b-4db6-8520-acc8abf2eb7a',
              'x': {'bdata': 'qJ6kQMOHokA=', 'dtype': 'f4'},
              'y': {'bdata': '9w+/QC29vkA=', 'dtype': 'f4'}},
             {'customdata': array(['694b430ed025705979bd9b81', '694b430ed025705979bd9b82',
                                   '694b430ed025705979bd9b83', '694b430ed025705979bd9b84'], dtype=object),
              'hovertemplate': ('<b>label: %{text}</b><br>x, y ' ... ': %{customdata}<extra></extra>'),
              'line': {'color': '#3283FE'}

✓ 图表已附加到 session


## 9. 使用说明

### Embeddings Panel 的主要功能：

1. **可视化 embeddings 空间**
   - 查看样本在 embedding 空间中的分布
   - 识别聚类和异常样本

2. **按字段着色**
   - 点击 "Color by" 下拉菜单
   - 选择 `ground_truth.label` 按标签着色
   - 可以查看不同标签的分布情况

3. **选择样本**
   - 使用鼠标拖拽（lasso tool）选择区域
   - 选中的样本会在主界面中高亮显示
   - 可以进一步分析或标记这些样本

4. **识别异常样本**
   - 查看远离主要聚类的点
   - 可能是标注错误或异常样本

5. **分析聚类结构**
   - 查看同一标签的样本是否聚集在一起
   - 识别可能混淆的类别

### 常见问题：

- **Q: 如何重新计算可视化？**  
  A: 重新运行步骤 6 的代码，使用相同的 `brain_key` 会覆盖之前的结果

- **Q: 如何保存选中的样本？**  
  A: 在 App 中选择样本后，可以使用标签功能标记这些样本

- **Q: 如何更改可视化方法？**  
  A: 修改 `method` 参数，可选值：`"umap"`, `"tsne"`, `"pca"` 等